# Third Normal Form (3NF)

After notebook 02 your Library is in First Normal Form: authors and genres
live in their own tables, and every value is atomic. But two tables still
carry redundancy driven by **transitive dependencies**.

- The `book` table stores `publisher_name`, `publisher_address`, and
  `publisher_city` on every book of the same publisher. The publisher's
  address is a fact about the publisher not the book.
- The `member` table stores `city` and `country` on every member. The
  country is a fact about the city not the member.

This notebook removes both redundancies by normalizing to **Third Normal
Form (3NF)** on the live Library. You should be continuing from where
notebook 02 left off.

In [ ]:
from rel_db_learning import (
    get_project_connection, run_sql, run_ddl, rebuild_messy_library, Relation,
)
from rel_db_learning.dbm_validator import validate_exercise
from rel_db_learning.dbm_validator.exercises import PROJECT_NORM_3NF
from rel_db_learning.widgets.self_assessment import multiple_choice, sql_query_check

project = get_project_connection()

## Restarting (if needed)

You should be continuing from notebook 02 at the 1NF state. If you ever
need to start fresh, run the cell below. It rebuilds the messy Stage 6
Library and applies the full 1NF migration in one step. Skip it if you
are already at the 1NF state.

In [ ]:
# Run only to restart at the 1NF state.
rebuild_messy_library()

run_ddl(r"""
    CREATE TABLE author (
        author_id serial PRIMARY KEY,
        name varchar
    );
    CREATE TABLE genre (
        genre_id serial PRIMARY KEY,
        name varchar
    );
    CREATE TABLE book_author (
        book_id   integer,
        author_id integer,
        PRIMARY KEY (book_id, author_id),
        CONSTRAINT book_author_book_fk   FOREIGN KEY (book_id)   REFERENCES book   (book_id),
        CONSTRAINT book_author_author_fk  FOREIGN KEY (author_id) REFERENCES author (author_id)
    );
    CREATE TABLE book_genre (
        book_id  integer,
        genre_id integer,
        PRIMARY KEY (book_id, genre_id),
        CONSTRAINT book_genre_book_fk   FOREIGN KEY (book_id)  REFERENCES book  (book_id),
        CONSTRAINT book_genre_genre_fk  FOREIGN KEY (genre_id) REFERENCES genre (genre_id)
    );
    INSERT INTO author (name)
        SELECT DISTINCT trim(a)
        FROM book, unnest(string_to_array(authors, ',')) AS a
        WHERE trim(a) <> '';
    INSERT INTO book_author (book_id, author_id)
        SELECT DISTINCT b.book_id, au.author_id
        FROM book b, unnest(string_to_array(b.authors, ',')) AS a
        JOIN author au ON au.name = trim(a)
        WHERE trim(a) <> '';
    INSERT INTO genre (name)
        SELECT DISTINCT trim(g)
        FROM book, unnest(string_to_array(genres, ',')) AS g
        WHERE trim(g) <> '';
    INSERT INTO book_genre (book_id, genre_id)
        SELECT DISTINCT b.book_id, ge.genre_id
        FROM book b, unnest(string_to_array(b.genres, ',')) AS g
        JOIN genre ge ON ge.name = trim(g)
        WHERE trim(g) <> '';
    ALTER TABLE book DROP COLUMN authors;
    ALTER TABLE book DROP COLUMN genres;
""", project)

## Third Normal Form: the theory

A **transitive dependency** occurs when a non-prime attribute depends on
another non-prime attribute, which in turn depends on the candidate key.
The chain looks like: `key -> B -> C`, where both B and C are non-prime.
C is not determined directly by the key but transitively through B.

**Third Normal Form (3NF):** a relation is in 3NF if and only if it is in
2NF and for every non-trivial functional dependency `X -> A`, either X is
a superkey or A is a prime attribute (part of some candidate key).

In other words, no non-prime attribute may be transitively determined by
a candidate key. Every non-key fact must depend directly on the key, the
whole key, and nothing but the key.

## Fix 1: the publisher transitive dependency

The book table stores the publisher's name, address, and city directly.
The dependency chain is:

```
book_id -> publisher_name -> publisher_address, publisher_city
```

`publisher_address` and `publisher_city` are facts about the publisher, not
the book. Any two books from the same publisher repeat the same address and
city, which is redundancy driven by the transitive dependency through
`publisher_name`.

In [ ]:
run_sql(r"""
    SELECT book_id, title, publisher_name, publisher_address, publisher_city
    FROM book
    ORDER BY publisher_name, book_id
    LIMIT 12
""", project)

In [ ]:
book_rel = Relation(
    ["book_id", "title", "publisher_name", "publisher_address", "publisher_city"],
    name="book",
)
book_rel.add_fd("book_id", ["title", "publisher_name", "publisher_address", "publisher_city"])
book_rel.add_fd("publisher_name", ["publisher_address", "publisher_city"])

print("is_3nf:     ", book_rel.is_3nf())
print("normal_form:", book_rel.normal_form())

The tool confirms the violation. The candidate key is `{book_id}`, so
`publisher_name`, `publisher_address`, and `publisher_city` are all non-prime.
The FD `publisher_name -> publisher_address, publisher_city` has a non-superkey
left side, making `publisher_address` and `publisher_city` transitively
dependent on `book_id` through `publisher_name`.

Because book has a single-attribute key, there are no partial dependencies,
so the table is in 2NF. The transitive dependency is the only problem,
which is why `normal_form()` returns `"2NF"` rather than `"1NF"`.

### The fix: extract the publisher table

The transitive dependency tells us that publisher facts belong in their
own table. We decompose into:

- `publisher(publisher_id, name, address, city)`: one row per publisher.
- `book(book_id, title, is_available, publisher_id)`: the FK replaces the
  three repeated columns.

Each publisher's address is now stored exactly once. Renaming a publisher's
city is a single update to one row in `publisher`. A new publisher can be
added before any of its books are catalogued.

### Step 1: create the publisher table and add the foreign key column

In pgModeler, open your 1NF design and Save As `stage_norm_3nf_design.dbm`.
Then:

1. Create table `publisher` with columns: `publisher_id` of type `serial`
   (set as primary key), `name` of type `varchar`, `address` of type
   `varchar`, `city` of type `varchar`.
2. Add column `publisher_id` of type `integer` (nullable) to the `book`
   table.
3. On the `book` table, add a foreign key constraint on `publisher_id`
   referencing `publisher(publisher_id)`. Use the Constraints tab on the
   book table, not the Relationship tool.
4. Save, then Database -> Diff against your live `my_project`. Save as **stage_norm_3nf_diff1.sql**. The Diff may include unrelated domain, index, or owner lines. Paste only the
   `CREATE TABLE publisher` statement, the `ALTER TABLE book ADD COLUMN
   publisher_id` line, and the `ALTER TABLE book ADD CONSTRAINT ... FOREIGN
   KEY` line. Ignore everything else.

In [ ]:
# Paste the DDL from your pgModeler Diff here:
publisher_schema_sql = r"""

"""
run_ddl(publisher_schema_sql, project)

In [ ]:
run_ddl(r"""
    INSERT INTO publisher (name, address, city)
    SELECT DISTINCT publisher_name, publisher_address, publisher_city
    FROM book;

    UPDATE book
    SET publisher_id = p.publisher_id
    FROM publisher p
    WHERE p.name = book.publisher_name;
""", project)

In [ ]:
# Confirm that every book now has a publisher_id. This should return 0.
run_sql(r"""
    SELECT COUNT(*) AS books_without_publisher
    FROM book
    WHERE publisher_id IS NULL
""", project)

### Step 2: remove the redundant publisher columns from book

Every book is now linked to its publisher through `publisher_id`. The
three repeated columns are no longer needed.

In pgModeler, remove `publisher_name`, `publisher_address`, and
`publisher_city` from the `book` table. Save, run Diff, and paste only
the three `ALTER TABLE book DROP COLUMN` statements below.

In [ ]:
# Paste the DROP COLUMN statements from your pgModeler Diff here:
publisher_drop_sql = r"""

"""
run_ddl(publisher_drop_sql, project)

In [ ]:
publisher_rel = Relation(
    ["publisher_id", "name", "address", "city"],
    name="publisher",
)
publisher_rel.add_fd("publisher_id", ["name", "address", "city"])

book_3nf_rel = Relation(
    ["book_id", "title", "publisher_id"],
    name="book",
)
book_3nf_rel.add_fd("book_id", ["title", "publisher_id"])

print("publisher  is_3nf:", publisher_rel.is_3nf(),  " normal_form:", publisher_rel.normal_form())
print("book       is_3nf:", book_3nf_rel.is_3nf(),   " normal_form:", book_3nf_rel.normal_form())

## Fix 2: the member city -> country transitive dependency

The member table stores the city and country of each member. The
dependency chain is:

```
member_id -> city -> country
```

`country` is a fact about the city, not the member. Every member in the
same city repeats the same country. An incorrect country on one row would
contradict another row with the same city.

In [ ]:
run_sql(r"""
    SELECT DISTINCT city, country
    FROM member
    ORDER BY country, city
""", project)

In [ ]:
member_rel = Relation(
    ["member_id", "full_name", "city", "country"],
    name="member",
)
member_rel.add_fd("member_id", ["full_name", "city", "country"])
member_rel.add_fd("city", "country")

print("is_3nf:     ", member_rel.is_3nf())
print("normal_form:", member_rel.normal_form())

The candidate key is `{member_id}`. Both `city` and `country` are
non-prime. The FD `city -> country` has a non-superkey left side, so
`country` is transitively determined by `member_id` through `city`.

As with the publisher case, the single-attribute key means there are no
partial dependencies (the relation is in 2NF). Only the transitive
dependency prevents 3NF.

### The fix: extract the city table

City and country facts belong together in their own table. We decompose
into:

- `city(city_id, name, country)`: one row per city.
- `member(member_id, full_name, address, email, phone, penalty_days,
  city_id)`: the FK replaces the two repeated columns.

Each city's country is now stored exactly once. Country is not given its
own table: it is a direct attribute of the city, not a separate entity
with additional facts.

### Step 3: create the city table and add the foreign key column

In pgModeler, with the 3NF design still open:

1. Create table `city` with columns: `city_id` of type `serial` (primary
   key), `name` of type `varchar`, `country` of type `varchar`.
2. Add column `city_id` of type `integer` (nullable) to the `member` table.
3. On the `member` table, add a foreign key constraint on `city_id`
   referencing `city(city_id)`. Use the Constraints tab, not the
   Relationship tool.
4. Save, Diff, and paste only the `CREATE TABLE city`, the `ALTER TABLE
   member ADD COLUMN city_id`, and the `ALTER TABLE member ADD CONSTRAINT
   ... FOREIGN KEY` lines below.

In [ ]:
# Paste the DDL from your pgModeler Diff here:
city_schema_sql = r"""

"""
run_ddl(city_schema_sql, project)

In [ ]:
run_ddl(r"""
    INSERT INTO city (name, country)
    SELECT DISTINCT city, country
    FROM member;

    UPDATE member
    SET city_id = c.city_id
    FROM city c
    WHERE c.name = member.city AND c.country = member.country;

""", project)

In [ ]:
# Confirm that every member now has a city_id. This should return 0.
run_sql(r"""
    SELECT COUNT(*) AS members_without_city
    FROM member
    WHERE city_id IS NULL
""", project)

### Step 4: remove city and country from member

Every member is now linked to their city through `city_id`. Remove the
old columns.

In pgModeler, remove `city` and `country` from the `member` table. Save,
Diff, and paste only the two `ALTER TABLE member DROP COLUMN` statements
below.

In [ ]:
# Paste the DROP COLUMN statements from your pgModeler Diff here:
city_drop_sql = r"""

"""
run_ddl(city_drop_sql, project)

In [ ]:
city_rel = Relation(
    ["city_id", "name", "country"],
    name="city",
)
city_rel.add_fd("city_id", ["name", "country"])

member_3nf_rel = Relation(
    ["member_id", "full_name", "address", "email", "phone", "penalty_days", "city_id"],
    name="member",
)
member_3nf_rel.add_fd(
    "member_id",
    ["full_name", "address", "email", "phone", "penalty_days", "city_id"],
)

print("city    is_3nf:", city_rel.is_3nf(),        " normal_form:", city_rel.normal_form())
print("member  is_3nf:", member_3nf_rel.is_3nf(),  " normal_form:", member_3nf_rel.normal_form())

## Validate the normalized Library

Both transitive dependencies have been removed. The Library now has 13
tables: the original 11 from the 1NF schema plus `publisher` and `city`.
Run the validator to confirm the full structure.

In [ ]:
validate_exercise(project_db=project, spec=PROJECT_NORM_3NF)

## The normalized schema in action

With `publisher` and `city` as proper tables, cross-table analytics become
clean joins rather than repeated-value aggregates.

In [ ]:
# How many books does each publisher city stock?
run_sql(r"""
    SELECT p.city, COUNT(*) AS books
    FROM publisher p
    JOIN book b ON b.publisher_id = p.publisher_id
    GROUP BY p.city
    ORDER BY books DESC, p.city
""", project)

In [ ]:
# How many members does the Library have per country?
run_sql(r"""
    SELECT c.country, COUNT(*) AS members
    FROM city c
    JOIN member m ON m.city_id = c.city_id
    GROUP BY c.country
    ORDER BY members DESC, c.country
""", project)

## Self-assessment

In [ ]:
multiple_choice(
    question="What is a transitive dependency?",
    options=[
        "A non-prime attribute that depends on only part of a composite candidate key",
        "A non-prime attribute that depends on another non-prime attribute, which depends on the key",
        "A dependency where two tables reference each other through foreign keys",
        "A trivial dependency of the form A -&gt; A",
    ],
    correct_index=1,
    explanation=(
        "A transitive dependency is when the chain key -&gt; B -&gt; C holds and B is"
        " not a superkey. In the original book table: book_id -&gt; publisher_name"
        " -&gt; publisher_address. Depending on part of a composite key is a"
        " partial dependency, which is a 2NF (not 3NF) violation."
    ),
)

In [ ]:
multiple_choice(
    question=(
        "Which functional dependency in the original book table causes the 3NF violation?"
    ),
    options=[
        "book_id -&gt; title, because title is a non-prime attribute",
        "book_id -&gt; publisher_name, because publisher_name is a non-prime attribute",
        "publisher_name -&gt; publisher_address, publisher_city, because publisher_name is not a superkey",
        "publisher_city -&gt; publisher_name, because city determines the publisher",
    ],
    correct_index=2,
    explanation=(
        "The FD publisher_name -&gt; publisher_address, publisher_city has a"
        " non-superkey left side. Since publisher_name is not a superkey of book"
        " and publisher_address/city are non-prime, this creates a transitive"
        " dependency violating 3NF. book_id -&gt; title and book_id -&gt; publisher_name"
        " are fine because the left side (book_id) is the key."
    ),
)

In [ ]:
multiple_choice(
    question=(
        "After the decomposition, why is the new book table (book_id, title, publisher_id) in 3NF?"
    ),
    options=[
        "Because book now has no functional dependencies at all",
        "Because the only non-trivial FD is book_id -&gt; title, publisher_id, and book_id is a superkey",
        "Because publisher_id is a foreign key and foreign keys always satisfy 3NF",
        "Because the publisher table stores address and city separately from name",
    ],
    correct_index=1,
    explanation=(
        "After removing publisher_name, publisher_address, and publisher_city from"
        " book, the only non-trivial dependency is book_id -&gt; title, publisher_id."
        " The left side book_id is the candidate key (a superkey), so every"
        " dependency has a superkey on the left. Both 3NF and BCNF are satisfied."
    ),
)

### Exercise: querying the normalized schema

Write a query that lists each publisher's name and the number of books
the Library holds from that publisher. Return only publishers with at
least one book, ordered by book count descending, then by publisher name.
You will need to join `publisher` and `book`.

In [ ]:
student_sql = r"""

"""
sql_query_check(
    student_sql=student_sql,
    correct_sql="m05_nb_04_exercise_1",
    connection=project,
)

## What's next

Your Library is now fully normalized to 3NF. Each fact lives in exactly
one place:

- Publisher name, address, and city are stored once per publisher.
- A city's country is stored once per city.
- Authors and genres are stored once per name, linked through junctions.

The next notebook covers **Boyce-Codd Normal Form (BCNF)**, a stricter
version of 3NF. Your Library's tables are already in BCNF, so notebook 05
uses a small constructed example to illustrate the difference. The pattern
you have seen in this module (find the dependency, decompose, confirm with
the tool) applies there too.